In [1]:
# packages
import duckdb
import pandas as pd


In [2]:
# load data
# updated this table to the merged topics table because I've clustered outlier articles
# need to join in the original articles df to get the clean_body text of articles.
con = duckdb.connect("../guardian_articles.duckdb")
df_merged = con.execute(
    """
    SELECT  merged.id,
            merged.webTitle,
            merged.webPublicationDate,
            merged.topic_id,
            merged.topic_prob,
            merged.topic_label,
            merged.source,
            clean.clean_body,
            clean.text
    FROM article_topics_merged merged
    JOIN article_topics clean ON merged.id = clean.id
    """
).df()

# checking the fields present in the article_topics table
# df_article_topics = con.execute(
#     """
#     SELECT  *
#     FROM article_topics
#     """
# ).df()

con.close()


In [3]:
df_merged.columns.tolist()
# df_article_topics.columns.tolist()

['id',
 'webTitle',
 'webPublicationDate',
 'topic_id',
 'topic_prob',
 'topic_label',
 'source',
 'clean_body',
 'text']

In [4]:
# identify top 20 topics
ranked_all_topics = df_merged["topic_label"].value_counts().reset_index()

ranked_all_topics.columns = ["topic", "count"]

In [5]:
print(ranked_all_topics)

                                             topic  count
0        0_prime minister_minister_australian_news  13766
1                                 -1_uncategorised   7883
2       1_premier league_liverpool_arsenal_chelsea    926
3    2_olympic_olympics_gold medal_winter olympics    136
4                 3_insomnia_sleep_waking_sleeping    110
..                                             ...    ...
112                      116_turkey_kurdish_and_of      6
113          110_bladder_urine_pelvic_incontinence      5
114                106_silica_dust_stone_silicosis      5
115              97_temperatures_snow_weather_cold      5
116                     99_venice_mose_city_lagoon      5

[117 rows x 2 columns]


In [6]:
# creating dataframe of just filtered articles with unclear topics so I can inspect and finish topic labelling

list_unclear_topic_ids = [
    44,
    55,
    58,
    60,
    61,
    64,
    68,
    71,
    72,
    74,
    76,
    77,
    78,
    79,
    83,
    84,
    87,
    88,
    89,
    90,
    93,
    94,
    99,
    102,
    104,
    105,
    109,
    111,
    112,
    113,
]

df_filtered = df_merged[df_merged["topic_id"].isin(list_unclear_topic_ids)]

In [7]:
print(df_filtered.columns.tolist())

['id', 'webTitle', 'webPublicationDate', 'topic_id', 'topic_prob', 'topic_label', 'source', 'clean_body', 'text']


In [8]:
# label topics
# 4/13 update: will need to update these once I have new merged in topics from the outliers subcorpus
TOPIC_LABEL_OVERRIDES = {
    -1: "Uncategorized",
    0: "Government and Politics",
    1: "Football & Soccer",
    2: "Olympics",
    3: "Sleep",
    4: "Christmas Trees",
    5: "Classical Music",
    6: "Formula 1",
    7: "Basketball",
    8: "Cosmetic Products",
    9: "Space Exploration",
    10: "Poetry",
    11: "Middle East",
    12: "Catholicism and the Pope",
    13: "Fertility and Reproductive Medicine",
    14: "Nicotine and the Tobacco Industry",
    15: "Crosswords and Puzzles",
    16: "The Beatles",
    17: "Skiing",
    18: "Astronomy",
    19: "School Shootings & Gun Control",
    20: "Scents and Fragrances",
    21: "Brazilian Politics",
    22: "Video Games",
    23: "Hurricanes",
    24: "Pets",
    25: "King Charles",
    26: "Baldness",
    27: "Bees",
    28: "Pandas",
    29: "Air Pollution",
    30: "Cannabis",
    31: "World Chess",
    32: "Boxing and Mixed Martial Arts",
    33: "Nuclear War",
    34: "Language",
    35: "Danish Camping",
    36: "Mount Vesuvius",
    37: "History of the Holocaust",
    38: "Scooters",
    39: "Quarterback Tom Brady",
    40: "Earbud Headphones",
    41: "Brain Cognition",
    42: "Rivers",
    43: "Italian Politics",
    44: "Saturday Quiz Game",
    45: "Extinct Mammoths and Genetic Engineering",
    46: "James Bond",
    47: "Health Insurance",
    48: "Portugal",
    49: "Marine Life",
    50: "Gut Health",
    51: "Baseball",
    52: "Mayor of New York Zohran Mamdani",
    53: "Blindness",
    54: "Abortion Policy",
    # 55:  this topic_id does not exist
    56: "Google Products",
    57: "Dinosaurs",
    58: "Night and Nocturnal Creatures",
    59: "Glaciers",
    60: "The NHS and British Healthcare",
    61: "Water Infrastructure",
    62: "Eye Health",
    63: "Bad Bunny's Super Bowl Halftime Performance",
    64: "Disability",
    65: "Jeffery Epstein Connection to British Royal Family",
    66: "Dogs as Pets",
    67: "Ireland",
    68: "Horror and Zombies",
    69: "Brain Diseases",
    70: "Eurovision",
    71: "Obituaries",
    72: "Cooking",
    73: "Reproductive Health",
    74: "Doctor Who",
    75: "Elvis Presley",
    76: "Miscellaneous Latin America",
    77: "Pacific Nation Politics",
    78: "Technology, AI, and Alcohol",
    79: "Wealth",
    80: "Electric Cars",
    81: "Solar Power",
    82: "Grenfell Tower Fire",
    83: "Missing Persons",
    84: "Baking",
    85: "Traditional Chinese Medicine and Alternative Therapies",
    86: "Libraries and Books",
    87: "Prince Andrew and Fraud Case",
    88: "Science Fiction",
    89: "AI and Data Centers",
    90: "Amazon and Indigenous Communities",
    91: "Gang Crime",
    92: "Star Trek",
    93: "Black Holes",
    94: "Plane Crashes",
    95: "Indonesia",
    96: "Diving",
    97: "Cold Temperatures",
    98: "Schools",
    99: "Venice",
    100: "Saving Money",
    101: "Major League Soccer",
    102: "Criminal Investigations",
    103: "Pac Man",
    104: "Maori Culture and Tattoos",
    105: "Oscar Wilde",
    106: "Silicon Dust",
    107: "Ticket Pricing",
    108: "Paris",
    109: "Peru and Bolivia",
    110: "Incontinence",
    111: "History of Medicine",
    112: "Miscellaneous Sports",
    113: "Arguments and Relationships",
    114: "South African Relations with the United States",
    115: "Medicare",
    116: "Turkey-Kurdish Relations",
}


In [9]:
df_merged.columns.tolist()

['id',
 'webTitle',
 'webPublicationDate',
 'topic_id',
 'topic_prob',
 'topic_label',
 'source',
 'clean_body',
 'text']

In [10]:
# create labelled table in DuckDB
df_merged["topic_label_clean"] = (
    df_merged["topic_id"].map(TOPIC_LABEL_OVERRIDES).fillna(df_merged["topic_id"])
)
# check if any topic_id values are not in the dictionary
missing = df_merged.loc[
    ~df_merged["topic_id"].isin(TOPIC_LABEL_OVERRIDES.keys()), "topic_id"
].unique()
print(
    f"The list of missing topic_ids that were not in the overrides dictionary: {missing}. If the list is empty the mapping was successful!"
)


# write labelled table to duckDB
conn = duckdb.connect("../guardian_articles.duckdb")
conn.execute(
    "CREATE OR REPLACE TABLE article_topics_labelled AS SELECT * FROM df_merged"
)
conn.close()

print(f"The labelled df has these columns: {df_merged.columns.tolist()}")
print(f"Written {len(df_merged):,} rows to article_topics_labelled")
print(f"Overrides applied: {len(TOPIC_LABEL_OVERRIDES)} topics")
print("The top 20 articles by count are:")
print(df_merged["topic_label_clean"].value_counts().head(20).to_string())


The list of missing topic_ids that were not in the overrides dictionary: []. If the list is empty the mapping was successful!
The labelled df has these columns: ['id', 'webTitle', 'webPublicationDate', 'topic_id', 'topic_prob', 'topic_label', 'source', 'clean_body', 'text', 'topic_label_clean']
Written 24,660 rows to article_topics_labelled
Overrides applied: 117 topics
The top 20 articles by count are:
topic_label_clean
Government and Politics                13766
Uncategorized                           7883
Football & Soccer                        926
Olympics                                 136
Sleep                                    110
Google Products                          106
Christmas Trees                           80
Classical Music                           47
Formula 1                                 36
Basketball                                34
Cosmetic Products                         34
The NHS and British Healthcare            33
Dinosaurs                          